In [12]:
import pandas as pd

def load_data(file_path, is_train=True):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(" ::: ")
            if is_train and len(parts) == 4:
                data.append(parts)
            elif not is_train and len(parts) == 3:
                data.append(parts)

    if is_train:
        df = pd.DataFrame(data, columns=["ID", "TITLE", "GENRE", "DESCRIPTION"])
    else:
        df = pd.DataFrame(data, columns=["ID", "TITLE", "DESCRIPTION"])

    return df

train_df = load_data("train_data.txt", is_train=True)
test_df = load_data("test_data.txt", is_train=False)

In [13]:
import re
from sklearn.model_selection import train_test_split

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

train_df["DESCRIPTION"] = train_df["DESCRIPTION"].apply(clean_text)
test_df["DESCRIPTION"] = test_df["DESCRIPTION"].apply(clean_text)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)
)

X = tfidf.fit_transform(train_df["DESCRIPTION"])
y = train_df["GENRE"]

In [15]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

LogisticRegression(max_iter=1000)

In [16]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X, y)

MultinomialNB()

In [17]:
from sklearn.svm import LinearSVC

model = LinearSVC()
model.fit(X, y)

LinearSVC()

In [18]:
X_test = tfidf.transform(test_df["DESCRIPTION"])
predictions = model.predict(X_test)

test_df["PREDICTED_GENRE"] = predictions

In [19]:
test_df[["ID", "PREDICTED_GENRE"]].to_csv("submission.csv", index=False)

In [20]:
import os
print(os.getcwd())

/content


In [21]:
# ---------- Prediction Function ----------
def predict_genre(text):
    text = clean_text(text)          # use same preprocessing
    vector = tfidf.transform([text]) # convert to TF-IDF
    prediction = model.predict(vector)[0]
    return prediction

# ---------- User Input ----------
user_input = input("Enter movie plot:\n")
genre = predict_genre(user_input)

print(f"Predicted Genre: {genre}")

Enter movie plot:
 A traumatized war pilot forced to land a commercial airliner when the crew gets food poisoning, surrounded by absurd characters.
Predicted Genre: comedy
